# 🛠️ 智聘Agent — 蓝领工人智能简历助手

---

## 项目背景

在中国，数以亿计的蓝领工人（工厂操作员、建筑工人、快递员、厨师、保洁员、家政人员等）在求职过程中面临一个共同的难题：**不会写简历**。

他们的困境主要体现在：

- **语言表达能力有限**：难以用书面语言准确描述自己的工作经验和技能
- **不了解简历格式**：不知道一份标准简历应该包含哪些模块、如何排版
- **无法有效使用 AI 工具**：即使有 Deepseek,豆包 等 AI 工具，也不知道该用什么提示词来获得帮助
- **缺乏润色能力**：口语化的表述无法转化为专业的简历用语

## 本工具的解决方案

**智聘Agent** 采用 **主动引导式对话** 的方式，像一个耐心的朋友一样，一步一步带着用户完成简历：

1. **无需用户思考"该写什么"** — AI 会主动提问，按照 教育背景 → 个人信息 → 工作经历 → 技能特长 → 个人评价 的顺序逐项引导
2. **用户只需要"说"就行** — 口语化描述即可，AI 自动将其润色为专业简历用语
3. **代写能力** — 用户说"我在工厂拧螺丝"，AI 能自动生成"负责产品组装流程，操作精密工具完成零部件装配，确保产品质量符合标准"
4. **标准模板输出** — 不求花哨的定制化，只输出 **标准通用简历模板**，四种配色可选，直接下载 PDF 即可投递
5. **实时预览** — 随时查看当前简历的制作进度

## 技术架构

```
用户（浏览器界面）
    ↕ 聊天交互
Flask 后端（本 Notebook）
    ↕ API 调用
火山引擎·豆包大模型（Doubao-1.5-pro-32k）
    ↕ 返回结构化 JSON
简历生成引擎（HTML → PDF）
```

## 功能清单

| 功能 | 说明 |
|------|------|
| 主动引导对话 | AI 按顺序逐项提问，用户无需思考该写什么 |
| 智能润色 | 口语化描述自动转为专业简历用语 |
| 代写能力 | 根据岗位和描述自动生成工作职责、项目描述 |
| 技能推断 | 根据工作经历自动推断并补充技能标签 |
| 实时预览 | 随时查看当前简历制作进度 |
| 四种模板 | 经典蓝、商务黑金、清新绿、极简灰 |
| PDF 导出 | 一键下载标准PDF格式简历 |
| 完成度追踪 | 顶部进度条实时显示简历完整度 |

---

## 第一步：安装依赖包

本项目需要以下 Python 包：

| 包名 | 用途 |
|------|------|
| `flask` | 轻量级 Web 后端框架，提供 API 接口 |
| `httpx` | 异步 HTTP 客户端，用于调用火山引擎大模型 API |
| `weasyprint` | 将 HTML 转换为 PDF 文件（服务端渲染） |

> **注意**：`weasyprint` 依赖系统级库。如果安装报错，请参考 [WeasyPrint 安装文档](https://doc.courtbouillon.org/weasyprint/stable/first_steps.html)。  
> Windows 用户可能需要安装 GTK3，或者改用 `pdfkit`（需安装 wkhtmltopdf）作为替代方案。  
> 如果实在装不了，后面的代码会自动降级为下载 HTML 文件。

In [ ]:
# ============================================================
# 安装项目所需的 Python 包
# flask: Web后端框架
# httpx: HTTP客户端，用于调用大模型API
# weasyprint: HTML转PDF（可选，装不上也能用）
# ============================================================

!pip install flask httpx

# weasyprint 单独安装，失败了也不影响主功能
try:
    import weasyprint
    print("✅ weasyprint 已安装")
except ImportError:
    print("⏳ 正在安装 weasyprint...")
    !pip install weasyprint
    try:
        import weasyprint
        print("✅ weasyprint 安装成功")
    except Exception as e:
        print(f"⚠️ weasyprint 安装失败（{e}），将使用 HTML 下载作为替代")
        print("   这不影响其他功能的使用")

---

## 第二步：配置大模型 API

本项目使用 **火山引擎·豆包大模型（Doubao-1.5-pro-32k）** 作为 AI 核心。

豆包大模型的 API 兼容 OpenAI 格式，只需要三个参数：
- `API_KEY`：你的火山引擎 API 密钥
- `BASE_URL`：火山引擎的 API 地址
- `MODEL`：模型的推理接入点 ID（Endpoint ID）

> 运行前请把火山引擎密钥写入环境变量 `ARK_API_KEY`。Notebook 不内置或提交真实 API Key。

In [ ]:
# ============================================================
# 大模型 API 配置
# 使用火山引擎（豆包）的 Chat Completions 接口
# 接口格式与 OpenAI 完全兼容
# ============================================================

# API 密钥（火山引擎控制台获取）
import os
API_KEY = os.getenv("ARK_API_KEY", "").strip()

# API 基础地址
BASE_URL = "https://ark.cn-beijing.volces.com/api/v3"

# 模型推理接入点（Endpoint ID）
# 当前使用：Doubao-1.5-pro-32k，适合长对话和复杂指令
MODEL = "ep-20260504194141-rgtsl"

print("✅ API 配置完成")
print(f"   模型接入点: {MODEL}")
print(f"   API 地址: {BASE_URL}")

---

## 第三步：定义简历数据结构

简历的所有信息存储在一个 Python 字典中，包含以下模块：

| 字段 | 类型 | 说明 |
|------|------|------|
| `name` | 字符串 | 姓名 |
| `phone` | 字符串 | 手机号 |
| `gender` | 字符串 | 性别 |
| `age` | 整数 | 年龄 |
| `city` | 字符串 | 目标工作城市 |
| `target_position` | 字符串 | 期望岗位 |
| `expected_salary` | 字符串 | 期望薪资 |
| `education` | 列表 | 教育背景（学校、学历、专业、时间） |
| `work_experiences` | 列表 | 工作/实习经历 |
| `projects` | 列表 | 项目经历 |
| `skills` | 列表 | 技能标签 |
| `awards` | 列表 | 获奖经历 |
| `self_eval` | 字符串 | 个人评价 |

AI 每次回复时都会返回这些字段的全量快照（snapshot），系统会自动合并更新。

In [ ]:
# ============================================================
# 简历数据结构定义
# 这是简历所有字段的「空白模板」
# AI 对话过程中会逐步填充这些字段
# ============================================================

def create_empty_resume():
    """创建一份空白简历数据结构"""
    return {
        "name": "",              # 姓名
        "phone": "",             # 手机号
        "gender": "",            # 性别
        "age": None,             # 年龄
        "city": "",              # 目标工作城市
        "target_position": "",   # 期望岗位
        "expected_salary": "",   # 期望薪资
        "education": [],         # 教育背景列表
        "work_experiences": [],   # 工作/实习经历列表
        "projects": [],          # 项目经历列表
        "skills": [],            # 技能标签列表
        "awards": [],            # 获奖经历列表
        "self_eval": ""          # 个人评价
    }


def update_resume(resume_data, snapshot):
    """
    将 AI 返回的快照数据合并到现有简历中。
    规则：只更新非空字段，不会覆盖已有数据为空值。
    
    参数:
        resume_data: 当前简历字典
        snapshot: AI 返回的简历字段快照
    返回:
        更新后的简历字典
    """
    if not snapshot or not isinstance(snapshot, dict):
        return resume_data
    
    for key, val in snapshot.items():
        # 跳过空值，避免覆盖已有数据
        if val is None or val == "" or (isinstance(val, list) and len(val) == 0):
            continue
        # 只更新简历中存在的字段
        if key in resume_data:
            resume_data[key] = val
    
    return resume_data


print("✅ 简历数据结构定义完成")

---

## 第四步：简历完成度计算

系统会根据当前已填写的字段，自动计算简历的完成度（0~100%）。

各模块的权重分配如下：

| 模块 | 权重 | 说明 |
|------|------|------|
| 基本信息 | 20% | 姓名、手机、性别、年龄、城市、岗位、薪资 |
| 教育背景 | 20% | 学校、学历、专业、时间 |
| 工作经历 | 25% | 公司、职位、时间、职责描述 |
| 项目经历 | 15% | 项目名、角色、描述 |
| 技能特长 | 10% | 技能标签数量 |
| 获奖经历 | 5% | 获奖条目 |
| 个人评价 | 5% | 评价文字长度 |

其中工作经历、项目经历、获奖经历、个人评价为可选模块，至少完成 2 个即可达到 75% 以上。

In [ ]:
# ============================================================
# 简历完成度计算引擎
# 根据各模块的填写情况计算 0-100 的完成度评分
# ============================================================

def calculate_completion(d):
    """
    计算简历完成百分比。
    
    参数:
        d: 简历数据字典
    返回:
        0-100 的整数，表示完成百分比
    """
    score = 0
    
    # ── 基本信息（满分20分）──
    basic = 0
    if d.get("name"):            basic += 4   # 姓名
    if d.get("phone"):           basic += 3   # 手机号
    if d.get("gender"):          basic += 2   # 性别
    if d.get("age"):             basic += 2   # 年龄
    if d.get("city"):            basic += 3   # 城市
    if d.get("target_position"): basic += 4   # 期望岗位
    if d.get("expected_salary"): basic += 2   # 期望薪资
    score += min(basic, 20)
    
    # ── 教育背景（满分20分）──
    edu = 0
    edu_list = d.get("education", [])
    if edu_list and isinstance(edu_list[0], dict):
        e = edu_list[0]
        if e.get("school"):   edu += 8   # 学校名称
        if e.get("degree"):   edu += 5   # 学历
        if e.get("major"):    edu += 4   # 专业
        if e.get("duration"): edu += 3   # 就读时间
    score += min(edu, 20)
    
    # ── 工作经历（满分25分）──
    exp = 0
    for w in d.get("work_experiences", []):
        if not isinstance(w, dict): continue
        e = 0
        if w.get("company") and w.get("position"): e += 3  # 公司+职位
        if w.get("duration"): e += 2                        # 时间
        resp_len = len(w.get("responsibilities", ""))
        if resp_len >= 80:   e += 10   # 详细的职责描述
        elif resp_len >= 40: e += 7
        elif resp_len >= 15: e += 4
        exp += min(e, 15)
    score += min(exp, 25)
    
    # ── 项目经历（满分15分）──
    proj = 0
    for p in d.get("projects", []):
        if not isinstance(p, dict): continue
        e = 0
        if p.get("name"): e += 2                           # 项目名
        if p.get("role") or p.get("duration"): e += 2     # 角色/时间
        desc_len = len(p.get("description", ""))
        if desc_len >= 80:   e += 6
        elif desc_len >= 40: e += 4
        elif desc_len >= 15: e += 2
        proj += min(e, 10)
    score += min(proj, 15)
    
    # ── 技能特长（满分10分）──
    sk = len([s for s in d.get("skills", []) if s])
    if sk >= 1: score += 3
    if sk >= 3: score += 4
    if sk >= 5: score += 3
    
    # ── 获奖经历（满分5分）──
    aw = [a for a in d.get("awards", []) if a and len(a) > 3]
    if aw: score += min(len(aw) * 2 + 1, 5)
    
    # ── 个人评价（满分5分）──
    eval_len = len((d.get("self_eval") or "").strip())
    if eval_len >= 15: score += 2
    if eval_len >= 60: score += 3
    
    return min(score, 100)


# 测试：空简历应该是 0%
test_resume = create_empty_resume()
print(f"✅ 完成度计算引擎就绪（空简历测试: {calculate_completion(test_resume)}%）")

---

## 第五步：构建 AI 系统提示词（System Prompt）

这是整个项目的**核心灵魂** — 系统提示词决定了 AI 的行为模式。

我们的提示词设计有几个关键点：

1. **主动引导**：AI 不会等用户说"帮我写简历"，而是直接按顺序提问
2. **代写优先**：用户只需要描述大概情况，AI 自动生成专业版本
3. **结构化输出**：AI 必须返回 JSON 格式，包含回复文本和简历数据快照
4. **STAR 法则**：引导用户用"情境-任务-行动-结果"的方式描述经历
5. **行业关键词注入**：根据目标岗位自动加入行业术语

> 为什么不直接让用户自己填表？因为我们的目标用户是蓝领工人，他们更适合**对话式**交互，而不是面对一堆空白表格。

In [ ]:
# ============================================================
# AI 系统提示词构建器
# 根据当前简历的填写状态，动态生成系统提示词
# 这样 AI 就能知道哪些信息已经收集、哪些还需要继续问
# ============================================================

def build_system_prompt(resume_data, completion):
    """
    根据当前简历状态构建系统提示词。
    
    参数:
        resume_data: 当前简历数据字典
        completion: 当前完成度百分比
    返回:
        完整的系统提示词字符串
    """
    rd = resume_data
    
    # ── 汇总已知信息，让 AI 了解当前进度 ──
    known_lines = []
    if rd["name"]:            known_lines.append(f'姓名: {rd["name"]}')
    if rd["gender"]:          known_lines.append(f'性别: {rd["gender"]}')
    if rd["age"]:             known_lines.append(f'年龄: {rd["age"]}')
    if rd["city"]:            known_lines.append(f'目标城市: {rd["city"]}')
    if rd["phone"]:           known_lines.append(f'手机: {rd["phone"]}')
    if rd["target_position"]: known_lines.append(f'期望岗位: {rd["target_position"]}')
    if rd["expected_salary"]: known_lines.append(f'期望薪资: {rd["expected_salary"]}')
    if rd["education"]:       known_lines.append(f'教育背景: {rd["education"]}')
    if rd["work_experiences"]:known_lines.append(f'工作/实习经历: {rd["work_experiences"]}')
    if rd["projects"]:        known_lines.append(f'项目经历: {rd["projects"]}')
    if rd["skills"]:          known_lines.append(f'技能: {rd["skills"]}')
    if rd["awards"]:          known_lines.append(f'竞赛获奖: {rd["awards"]}')
    if rd["self_eval"]:       known_lines.append(f'个人评价: {rd["self_eval"]}')
    
    known = "\n".join(known_lines) if known_lines else "（尚未收集到任何信息）"
    tp = rd["target_position"] or ""
    
    return f"""你是一个专业的简历制作助手，帮助用户制作标准的中文简历。

【已知信息（完成度 {completion}%）】
{known}

【简历模板结构（按此顺序收集）】
① 基本信息：姓名、性别、年龄、城市、手机、期望岗位、期望薪资
② 教育背景：学校、学历、专业、就读时间
③ 实习/工作经历：公司、职位、时间、职责（代写确认）
④ 项目经历：项目名、角色、时间、描述（代写确认）
⑤ 技能特长：技能标签列表（自动推断确认）
⑥ 竞赛获奖：获奖列表
⑦ 个人评价：综合素质段落（代写确认）

注意：③④⑥⑦ 是可选模块，用户只需要有其中 2 个以上就算完整

【核心行为准则】
▌原则1：代写优先 — 用户描述工作职责、项目、个人评价时，立即代写完整专业版本，用「」包裹展示
▌原则2：技能自动推断 — 根据岗位和经历自动推断技能标签
▌原则3：引导可选模块 — 没有工作经历则问项目/竞赛
▌原则4：一次只推进一步
▌原则5：STAR 法则引导
▌原则6：行业关键词注入（岗位：{tp}）
▌原则7：润色前后对比

【输出格式 — 必须是且仅是合法 JSON】
{{"reply":"给用户的话","snapshot":{{"name":"","gender":"","age":null,"city":"","phone":"","target_position":"","expected_salary":"","education":[],"work_experiences":[],"projects":[],"skills":[],"awards":[],"self_eval":""}},"completion":0,"ready":false,"quick_replies":["好的","帮我写","跳过这一项","没有了"]}}

【字段规则】
- snapshot 是全量快照，历史已知字段必须全部保留
- responsibilities、description 用中文分号分隔，禁止换行符
- completion 估算：基本信息20 + 教育20 + 工作25 + 项目15 + 技能10 + 获奖5 + 评价5
- ready: completion >= 75 且可选模块至少2个已填
- quick_replies: 2-4个快捷回复

【收集顺序】教育背景 → 姓名/性别/年龄 → 城市/手机 → 期望岗位/薪资 → 实习经历 → 项目经历 → 技能 → 竞赛获奖 → 个人评价"""


print("✅ 系统提示词构建器就绪")

---

## 第六步：AI 对话服务（后端核心）

这个模块负责：
1. 将用户消息和历史对话发送给豆包大模型
2. 接收 AI 的 JSON 回复
3. 解析出回复文本（`reply`）和简历数据快照（`snapshot`）

AI 返回的 JSON 可能不太规范（比如多余的换行、markdown 代码块包裹等），所以我们需要做**容错解析**：先尝试标准 JSON 解析，失败了就用正则提取关键字段。

In [ ]:
# ============================================================
# AI 对话服务
# 负责调用豆包大模型 API 并解析返回结果
# ============================================================

import httpx
import json
import re


def parse_ai_response(raw_text):
    """
    解析 AI 返回的原始文本，提取 JSON 数据。
    AI 有时会在 JSON 前后加 markdown 代码块标记，需要清理。
    
    参数:
        raw_text: AI 返回的原始字符串
    返回:
        字典，包含 reply, snapshot, completion, ready, quick_replies
    """
    raw = raw_text.strip()
    
    # 去掉 markdown 代码块包裹（```json ... ```）
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:])
    if raw.endswith("```"):
        raw = "\n".join(raw.split("\n")[:-1])
    raw = raw.strip()
    
    # 找到 JSON 的起止位置
    start = raw.find("{")
    end = raw.rfind("}") + 1
    
    if start >= 0 and end > start:
        try:
            # 清理 JSON 中的换行符（AI 有时会在字符串值中插入换行）
            json_str = raw[start:end]
            json_str = re.sub(r'[\r\n]+', ' ', json_str)
            parsed = json.loads(json_str)
            
            return {
                "reply": str(parsed.get("reply", "好的，咱们继续～")),
                "snapshot": parsed.get("snapshot") or parsed.get("updates") or {},
                "completion": int(parsed.get("completion", 0)),
                "ready": bool(parsed.get("ready", False)),
                "quick_replies": parsed.get("quick_replies", [])
            }
        except json.JSONDecodeError:
            # JSON 解析失败，尝试用正则提取 reply 字段
            match = re.search(r'"reply"\s*:\s*"((?:[^"\\]|\\.)*)"', raw)
            if match:
                reply = match.group(1).replace("\\n", "\n").replace('\\"', '"')
                return {"reply": reply, "snapshot": {}, "completion": 0, "ready": False, "quick_replies": []}
    
    # 兜底：直接返回原始文本作为回复
    return {
        "reply": raw[:200] or "好的，咱们继续～",
        "snapshot": {},
        "completion": 0,
        "ready": False,
        "quick_replies": []
    }


def call_ai(messages, resume_data, completion):
    """
    调用豆包大模型 API，获取 AI 回复。
    
    参数:
        messages: 对话历史列表 [{"role": "user/assistant", "content": "..."}]
        resume_data: 当前简历数据
        completion: 当前完成度
    返回:
        解析后的结果字典
    """
    # 构建系统提示词
    system_prompt = build_system_prompt(resume_data, completion)
    
    # 组装 API 请求的消息列表
    api_messages = [{"role": "system", "content": system_prompt}]
    
    # 只取最近 20 条对话，避免超出模型上下文长度
    recent = messages[-20:]
    for m in recent:
        if m["role"] in ("user", "assistant"):
            api_messages.append({"role": m["role"], "content": m["content"]})
    
    # 发送 HTTP 请求到火山引擎
    response = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL,
            "messages": api_messages,
            "temperature": 0.6,   # 控制创造性，0.6 兼顾稳定和灵活
            "max_tokens": 1800    # 限制回复长度
        },
        timeout=30  # 超时时间30秒
    )
    
    # 检查 HTTP 状态码
    if response.status_code != 200:
        raise Exception(f"API 返回错误 {response.status_code}: {response.text[:200]}")
    
    # 提取 AI 回复文本
    data = response.json()
    raw_text = data["choices"][0]["message"]["content"]
    
    # 解析 JSON 结果
    return parse_ai_response(raw_text)


print("✅ AI 对话服务就绪")

---

## 第七步：简历 HTML 模板引擎

本项目内置 **4 种简历模板**，通过 CSS 变量（CSS Custom Properties）实现配色切换：

| 模板 ID | 名称 | 风格 |
|---------|------|------|
| `classic_blue` | 经典蓝 | 通用求职，蓝色主题 |
| `business_dark` | 商务黑金 | 偏正式，双线边框 |
| `fresh_green` | 清新绿 | 年轻活力，虚线装饰 |
| `minimal_gray` | 极简灰 | 低调简约，适合技术岗 |

模板引擎会根据简历数据自动判断哪些模块有内容、哪些为空，只渲染有数据的模块，避免出现空白区域。

In [ ]:
# ============================================================
# 简历 HTML 模板引擎
# 将简历数据渲染为带样式的 HTML 页面
# 支持 4 种配色模板，通过 CSS 变量切换
# ============================================================

import html as html_lib  # 用于 HTML 转义，防止 XSS
from datetime import datetime

# ── 四种模板的 CSS 变量定义 ──
TEMPLATES = {
    "classic_blue": ":root{--primary:#1a73e8;--primary-light:#d2e3fc;--bg-body:#f8f9fa;--bg-page:#fff;--text-main:#2c2c2c;--text-sec:#555;--text-muted:#999;--hdr-bw:2.5px;--hdr-bs:solid;--sec-bw:1.5px;--sec-bs:solid;--hdr-align:center;--name-w:800;--name-s:26px;--avatar-bg:#1a73e8}",
    "business_dark": ":root{--primary:#2c3e50;--primary-light:#bdc3c7;--bg-body:#f8f9fa;--bg-page:#fff;--text-main:#2c2c2c;--text-sec:#4a4a4a;--text-muted:#888;--hdr-bw:3px;--hdr-bs:double;--sec-bw:1px;--sec-bs:solid;--hdr-align:left;--name-w:900;--name-s:24px;--avatar-bg:#2c3e50}.r-header{border-bottom-color:#e67e22!important}",
    "fresh_green": ":root{--primary:#27ae60;--primary-light:#d5f5e3;--bg-body:#f8f9fa;--bg-page:#fff;--text-main:#2c3e2c;--text-sec:#557755;--text-muted:#88aa88;--hdr-bw:2px;--hdr-bs:solid;--sec-bw:1.5px;--sec-bs:dashed;--hdr-align:center;--name-w:700;--name-s:26px;--avatar-bg:#27ae60}",
    "minimal_gray": ":root{--primary:#444;--primary-light:#ddd;--bg-body:#f8f9fa;--bg-page:#fff;--text-main:#333;--text-sec:#666;--text-muted:#aaa;--hdr-bw:1px;--hdr-bs:solid;--sec-bw:1px;--sec-bs:solid;--hdr-align:left;--name-w:600;--name-s:22px;--avatar-bg:#444}"
}


def esc(s):
    """HTML 转义，防止注入攻击"""
    return html_lib.escape(str(s)) if s else ""


def format_bullets(raw):
    """
    将分号分隔的职责/描述文本转换为 HTML 列表。
    例如："负责A；负责B；负责C" → <ul><li>负责A</li>...</ul>
    """
    if not raw:
        return ""
    # 统一全角/半角分号，然后按分号分割
    lines = [l.strip() for l in raw.replace("；", ";").split(";") if l.strip()]
    if not lines:
        return f'<p style="font-size:12px;line-height:1.8">{esc(raw)}</p>'
    items = "".join(f"<li>{esc(l)}</li>" for l in lines)
    return f'<ul style="padding-left:16px;font-size:12px;line-height:1.8;list-style:disc">{items}</ul>'


print("✅ 模板工具函数就绪")

In [ ]:
# ============================================================
# 简历 HTML 完整生成函数
# 将简历数据 + 模板配色组合为一份完整的 HTML 页面
# 该 HTML 可以直接在浏览器中打开，也可以转换为 PDF
# ============================================================

def generate_resume_html(resume_data, template_id="classic_blue"):
    """
    生成完整的简历 HTML 页面。
    
    参数:
        resume_data: 简历数据字典
        template_id: 模板ID（classic_blue/business_dark/fresh_green/minimal_gray）
    返回:
        完整的 HTML 字符串
    """
    d = resume_data
    css = TEMPLATES.get(template_id, TEMPLATES["classic_blue"])
    
    # ── 基本信息 ──
    name = esc(d.get("name") or "姓名")
    phone = esc(d.get("phone") or "")
    gender = esc(d.get("gender") or "")
    age = d.get("age")
    city = esc(d.get("city") or "")
    tp = esc(d.get("target_position") or "")
    es = esc(d.get("expected_salary") or "")
    initial = (d.get("name") or "U")[0].upper()  # 头像首字母
    
    # 日期戳
    now = datetime.now()
    date_str = f"{now.year}年{now.month}月{now.day}日"
    
    # ── 联系方式行 ──
    contact_parts = []
    if phone:  contact_parts.append(f"📞 {phone}")
    if gender: contact_parts.append(gender)
    if age:    contact_parts.append(f"{age}岁")
    if city:   contact_parts.append(f"📍 {city}")
    contact_html = ' <span style="color:#bbb;margin:0 4px">|</span> '.join(contact_parts)
    
    # ── 求职意向 ──
    ti_html = ""
    ti_parts = []
    if tp: ti_parts.append(f"期望岗位：<b>{tp}</b>")
    if es: ti_parts.append(f"期望薪资：<b>{es}</b>")
    if ti_parts:
        ti_content = ' <span style="color:#bbb;margin:0 8px">｜</span> '.join(ti_parts)
        ti_html = f'<div class="r-section"><h2 class="r-title">求职意向</h2><div style="font-size:13px">{ti_content}</div></div>'
    
    # ── 教育背景 ──
    edu_html = ""
    edu_list = d.get("education", [])
    if edu_list:
        rows = ""
        for e in edu_list:
            if not isinstance(e, dict): continue
            rows += f'''<div style="margin-bottom:12px">
                <div style="display:flex;justify-content:space-between">
                    <b style="font-size:13px">{esc(e.get("school", ""))}</b>
                    <span style="font-size:11px;color:var(--text-muted)">{esc(e.get("duration", ""))}</span>
                </div>
                <div style="font-size:12px;color:var(--text-sec)">
                    {" · ".join(filter(None, [esc(e.get("degree", "")), esc(e.get("major", ""))]))}
                </div>
            </div>'''
        if rows:
            edu_html = f'<div class="r-section"><h2 class="r-title">教育背景</h2>{rows}</div>'
    
    # ── 工作经历 ──
    work_html = ""
    work_list = d.get("work_experiences", [])
    if work_list:
        rows = ""
        for w in work_list:
            if not isinstance(w, dict): continue
            rows += f'''<div style="margin-bottom:14px">
                <div style="display:flex;justify-content:space-between">
                    <b style="font-size:13px">{esc(w.get("company", ""))}</b>
                    <span style="font-size:11px;color:var(--text-muted)">{esc(w.get("duration", ""))}</span>
                </div>
                <div style="font-size:12px;color:var(--text-sec);margin-bottom:4px">{esc(w.get("position", ""))}</div>
                {format_bullets(w.get("responsibilities", ""))}
            </div>'''
        if rows:
            work_html = f'<div class="r-section"><h2 class="r-title">实习 / 工作经历</h2>{rows}</div>'
    
    # ── 项目经历 ──
    proj_html = ""
    proj_list = d.get("projects", [])
    if proj_list:
        rows = ""
        for p in proj_list:
            if not isinstance(p, dict): continue
            meta = " · ".join(filter(None, [esc(p.get("role", "")), esc(p.get("duration", ""))]))
            rows += f'''<div style="margin-bottom:14px">
                <div style="display:flex;justify-content:space-between">
                    <b style="font-size:13px">{esc(p.get("name", ""))}</b>
                    <span style="font-size:11px;color:var(--text-muted)">{meta}</span>
                </div>
                {format_bullets(p.get("description", ""))}
            </div>'''
        if rows:
            proj_html = f'<div class="r-section"><h2 class="r-title">项目经历</h2>{rows}</div>'
    
    # ── 技能特长 ──
    skill_html = ""
    skills = [s for s in d.get("skills", []) if s]
    if skills:
        items = "".join(f"<li>{esc(s)}</li>" for s in skills)
        skill_html = f'<div class="r-section"><h2 class="r-title">技能特长</h2><ul style="padding-left:16px;font-size:12px;line-height:1.8;list-style:disc">{items}</ul></div>'
    
    # ── 获奖经历 ──
    award_html = ""
    awards = [a for a in d.get("awards", []) if a]
    if awards:
        items = "".join(f"<li>{esc(a)}</li>" for a in awards)
        award_html = f'<div class="r-section"><h2 class="r-title">竞赛获奖</h2><ul style="padding-left:16px;font-size:12px;line-height:1.9;list-style:disc">{items}</ul></div>'
    
    # ── 个人评价 ──
    eval_html = ""
    if d.get("self_eval"):
        eval_html = f'<div class="r-section"><h2 class="r-title">个人评价</h2><p style="font-size:12.5px;line-height:1.8;text-align:justify">{esc(d["self_eval"])}</p></div>'
    
    # ── 组装完整 HTML ──
    return f"""<!DOCTYPE html>
<html lang="zh-CN"><head><meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{name}-简历</title>
<style>
{css}
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:"PingFang SC","Microsoft YaHei",Arial,sans-serif;font-size:13px;color:var(--text-main);background:var(--bg-body);-webkit-print-color-adjust:exact}}
.page{{max-width:830px;margin:10px auto 30px;background:var(--bg-page);padding:32px 28px;box-shadow:0 4px 20px rgba(0,0,0,.1);border-radius:6px}}
.r-header{{text-align:var(--hdr-align);padding-bottom:14px;margin-bottom:20px;border-bottom:var(--hdr-bw) var(--hdr-bs) var(--primary);display:flex;align-items:center;gap:14px}}
.r-header-text{{flex:1;text-align:var(--hdr-align)}}
.r-name{{font-size:var(--name-s);font-weight:var(--name-w);letter-spacing:4px;color:#111;margin-bottom:6px}}
.r-contact{{font-size:12px;color:var(--text-sec);line-height:1.8}}
.avatar-w{{width:48px;height:48px;border-radius:50%;flex-shrink:0;display:flex;align-items:center;justify-content:center;background:var(--avatar-bg);font-size:22px;font-weight:700;color:#fff}}
.r-section{{margin-bottom:18px}}
.r-title{{font-size:13px;font-weight:700;color:var(--primary);border-bottom:var(--sec-bw) var(--sec-bs) var(--primary-light);padding-bottom:4px;margin-bottom:10px;letter-spacing:1px}}
.r-footer{{margin-top:24px;text-align:right;font-size:10px;color:#ccc}}
@media print{{body{{background:#fff}}.page{{margin:0;box-shadow:none;padding:28px 36px}}}}
</style></head><body>
<div class="page">
    <div class="r-header">
        <div class="avatar-w">{esc(initial)}</div>
        <div class="r-header-text">
            <div class="r-name">{name}</div>
            <div class="r-contact">{contact_html}</div>
        </div>
    </div>
    {ti_html}{edu_html}{work_html}{proj_html}{skill_html}{award_html}{eval_html}
    <div class="r-footer">由智聘Agent生成 · {date_str}</div>
</div></body></html>"""


print("✅ 简历 HTML 模板引擎就绪")

---

## 第八步：PDF 导出服务

将生成的简历 HTML 转换为 A4 尺寸的 PDF 文件。

优先使用 `weasyprint`（服务端渲染，效果最好），如果不可用则降级为保存 HTML 文件。

导出的 PDF 文件名格式为：`姓名_2026-05-06.pdf`

In [ ]:
# ============================================================
# PDF 导出服务
# 将简历 HTML 转换为 PDF 文件
# 支持 weasyprint（优先）和 HTML 下载（降级方案）
# ============================================================

import os

# 检测 weasyprint 是否可用
WEASYPRINT_AVAILABLE = False
try:
    import weasyprint
    WEASYPRINT_AVAILABLE = True
except ImportError:
    pass


def export_resume(resume_data, template_id="classic_blue", output_dir="."):
    """
    导出简历文件。
    
    参数:
        resume_data: 简历数据字典
        template_id: 模板ID
        output_dir: 输出目录
    返回:
        (文件路径, 文件类型) 元组
    """
    # 生成简历 HTML
    resume_html = generate_resume_html(resume_data, template_id)
    
    # 文件名
    name = resume_data.get("name") or "简历"
    date_str = datetime.now().strftime("%Y-%m-%d")
    
    if WEASYPRINT_AVAILABLE:
        # 使用 weasyprint 生成 PDF
        pdf_path = os.path.join(output_dir, f"{name}_{date_str}.pdf")
        html_doc = weasyprint.HTML(string=resume_html)
        html_doc.write_pdf(pdf_path)
        return pdf_path, "pdf"
    else:
        # 降级方案：保存为 HTML 文件
        html_path = os.path.join(output_dir, f"{name}_{date_str}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(resume_html)
        return html_path, "html"


if WEASYPRINT_AVAILABLE:
    print("✅ PDF 导出服务就绪（使用 weasyprint）")
else:
    print("⚠️ weasyprint 不可用，将使用 HTML 下载作为替代")
    print("   提示：前端界面中仍然可以通过浏览器打印功能另存为 PDF")

---

## 第九步：Flask 后端路由

Flask 后端提供以下 API 接口：

| 路由 | 方法 | 功能 |
|------|------|------|
| `/` | GET | 返回前端聊天界面 HTML |
| `/api/chat` | POST | 接收用户消息，调用 AI 并返回回复 |
| `/api/preview` | POST | 生成简历预览 HTML |
| `/api/download` | POST | 导出并下载简历 PDF |
| `/api/reset` | POST | 重置对话和简历数据 |

所有状态（对话历史、简历数据）都存储在服务器内存中，重启后会丢失。这对于单用户本地使用完全够用。

In [ ]:
# ============================================================
# Flask 后端应用
# 提供 API 接口，连接前端界面和 AI 服务
# ============================================================

from flask import Flask, request, jsonify, send_file, Response
import threading

# 创建 Flask 应用实例
app = Flask(__name__)

# ── 全局状态（单用户模式）──
# 存储在服务器内存中，重启后丢失
chat_messages = []                          # 对话历史
resume_data = create_empty_resume()         # 简历数据
current_completion = 0                      # 当前完成度

# AI 的开场白
GREETING = """你好！我是你的简历助手 \n我会帮你制作一份专业简历——内容越丰富，机会越多！\n\n先从教育背景开始，这个放在简历最显眼的地方。\n你是什么学历，在哪所学校就读或毕业的？"""


print("✅ Flask 应用实例创建完成")

In [ ]:
# ============================================================
# API 路由：聊天接口
# POST /api/chat — 接收用户消息，调用 AI，返回回复
# ============================================================

@app.route("/api/chat", methods=["POST"])
def api_chat():
    """
    聊天接口：
    请求体: {"message": "用户输入的文本"}
    返回: {"reply": "AI回复", "completion": 35, "quick_replies": [...]}
    """
    global resume_data, current_completion
    
    data = request.get_json()
    user_msg = data.get("message", "").strip()
    if not user_msg:
        return jsonify({"error": "消息不能为空"}), 400
    
    # 记录用户消息
    chat_messages.append({"role": "user", "content": user_msg})
    
    try:
        # 调用 AI
        result = call_ai(chat_messages, resume_data, current_completion)
        
        # 更新简历数据
        resume_data = update_resume(resume_data, result.get("snapshot"))
        
        # 更新完成度（取 AI 估算和本地计算的较大值）
        local_comp = calculate_completion(resume_data)
        ai_comp = result.get("completion", 0)
        current_completion = max(local_comp, ai_comp)
        
        # 记录 AI 回复
        reply = result.get("reply", "好的，咱们继续～")
        chat_messages.append({"role": "assistant", "content": reply})
        
        return jsonify({
            "reply": reply,
            "completion": current_completion,
            "quick_replies": result.get("quick_replies", []),
            "ready": result.get("ready", False)
        })
        
    except Exception as e:
        return jsonify({"error": f"AI 服务出错：{str(e)}"}), 500


print("✅ 聊天接口注册完成: POST /api/chat")

In [ ]:
# ============================================================
# API 路由：简历预览、下载、重置
# ============================================================

@app.route("/api/preview", methods=["POST"])
def api_preview():
    """
    简历预览接口：
    请求体: {"template": "classic_blue"}
    返回: {"html": "简历HTML字符串", "completion": 35}
    """
    data = request.get_json() or {}
    template_id = data.get("template", "classic_blue")
    
    html = generate_resume_html(resume_data, template_id)
    return jsonify({
        "html": html,
        "completion": current_completion
    })


@app.route("/api/download", methods=["POST"])
def api_download():
    """
    简历下载接口：
    请求体: {"template": "classic_blue"}
    返回: PDF 或 HTML 文件
    """
    data = request.get_json() or {}
    template_id = data.get("template", "classic_blue")
    
    # 检查是否有内容
    if not any([resume_data.get("name"), resume_data.get("education"),
                resume_data.get("work_experiences"), resume_data.get("skills")]):
        return jsonify({"error": "还没有简历内容可下载"}), 400
    
    file_path, file_type = export_resume(resume_data, template_id)
    mime = "application/pdf" if file_type == "pdf" else "text/html"
    return send_file(file_path, mimetype=mime, as_attachment=True)


@app.route("/api/reset", methods=["POST"])
def api_reset():
    """
    重置接口：清空对话和简历数据，重新开始
    """
    global resume_data, current_completion
    chat_messages.clear()
    resume_data = create_empty_resume()
    current_completion = 0
    return jsonify({"status": "ok", "greeting": GREETING})


print("✅ 预览/下载/重置接口注册完成")
print("   POST /api/preview  — 简历预览")
print("   POST /api/download — 简历下载")
print("   POST /api/reset    — 重置对话")

---

## 第十步：前端聊天界面

前端采用 **微信风格聊天界面**，所有代码内嵌在一个 HTML 字符串中，由 Flask 的 `/` 路由直接返回。

界面特点：
- 绿色微信风格配色
- 气泡式对话消息（AI 白色、用户绿色）
- 顶部进度条实时显示简历完成度
- 快捷回复按钮，降低输入门槛
- 工具栏：预览简历、下载 PDF、重新开始
- 底部弹出式简历预览模态框
- 语音输入按钮（使用 Web Speech API）
- 自动朗读开关（使用 Speech Synthesis API）
- 完全适配手机屏幕

前端通过 `fetch` 调用后端的 `/api/*` 接口，与上面定义的 Flask 路由配合工作。

> 与纯前端版本不同，这里的 API Key 存储在后端（Python 进程中），不会暴露给浏览器用户。

In [ ]:
# ============================================================
# 前端 HTML 界面（微信风格聊天界面）
# 所有 HTML/CSS/JS 内嵌在一个 Python 字符串中
# 由 Flask 根路由 "/" 直接返回给浏览器
# ============================================================

FRONTEND_HTML = r"""
<!DOCTYPE html>
<html lang="zh-CN">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
<title>智聘Agent - AI简历助手</title>
<style>
/* ══════ 全局重置 ══════ */
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{
  font-family:-apple-system,"PingFang SC","Microsoft YaHei","Helvetica Neue",Arial,sans-serif;
  background:#ededed;color:#333;font-size:15px;
  -webkit-tap-highlight-color:transparent;
  overflow:hidden;height:100vh;
}

/* ══════ 布局容器 ══════ */
.app{display:flex;flex-direction:column;height:100vh;max-width:500px;margin:0 auto;background:#ededed;position:relative}

/* ── 顶部栏 ── */
.header{
  background:#07c160;color:#fff;
  padding:12px 16px;display:flex;align-items:center;justify-content:space-between;
  flex-shrink:0;z-index:10;
}
.header-left{display:flex;align-items:center;gap:10px}
.header-title{font-size:17px;font-weight:600}
.header-right{display:flex;align-items:center;gap:6px;font-size:12px}
.progress-bar{width:50px;height:4px;background:rgba(255,255,255,.4);border-radius:2px;overflow:hidden}
.progress-fill{height:100%;background:#fff;border-radius:2px;transition:width .5s ease}

/* ── 自动朗读开关 ── */
.tts-toggle{
  display:flex;align-items:center;gap:4px;
  font-size:12px;color:#fff;cursor:pointer;
  padding:4px 8px;border-radius:12px;
  background:rgba(255,255,255,.15);
  transition:all .2s;user-select:none;
}
.tts-toggle.active{background:rgba(255,255,255,.35)}
.tts-switch{
  width:28px;height:16px;background:rgba(255,255,255,.35);
  border-radius:8px;position:relative;transition:all .2s;flex-shrink:0;
}
.tts-switch::after{
  content:"";position:absolute;left:2px;top:2px;
  width:12px;height:12px;background:#fff;border-radius:50%;
  transition:transform .2s;
}
.tts-toggle.active .tts-switch{background:#fff}
.tts-toggle.active .tts-switch::after{transform:translateX(12px);background:#07c160}

/* ── 工具栏 ── */
.toolbar{
  display:flex;gap:8px;padding:8px 12px;background:#f7f7f7;
  border-bottom:1px solid #e0e0e0;flex-shrink:0;
}
.toolbar-btn{
  flex:1;padding:8px 0;border:1px solid #ddd;border-radius:8px;
  background:#fff;color:#333;font-size:13px;text-align:center;
  cursor:pointer;transition:all .2s;display:flex;align-items:center;justify-content:center;gap:4px;
}
.toolbar-btn:active{background:#07c160;color:#fff;border-color:#07c160}

/* ── 聊天区 ── */
.chat-area{flex:1;overflow-y:auto;padding:12px;-webkit-overflow-scrolling:touch}

/* ── 消息气泡 ── */
.msg{display:flex;margin-bottom:14px;gap:8px;align-items:flex-start}
.msg.user{flex-direction:row-reverse}
.msg .avatar{
  width:38px;height:38px;border-radius:6px;flex-shrink:0;
  display:flex;align-items:center;justify-content:center;
  font-size:14px;font-weight:600;color:#fff;
}
.msg.assistant .avatar{background:#07c160}
.msg.user .avatar{background:#1989fa}
.msg .bubble{
  max-width:72%;padding:10px 12px;font-size:15px;line-height:1.6;
  word-break:break-word;position:relative;
}
.msg.assistant .bubble{background:#fff;border-radius:4px 16px 16px 16px;box-shadow:0 1px 2px rgba(0,0,0,.06)}
.msg.user .bubble{background:#95ec69;border-radius:16px 4px 16px 16px;box-shadow:0 1px 2px rgba(0,0,0,.06)}

/* ── 快捷回复 ── */
.quick-replies{display:flex;flex-wrap:wrap;gap:8px;padding:4px 12px 10px}
.quick-btn{
  padding:6px 14px;border:1px solid #07c160;border-radius:16px;
  background:#fff;color:#07c160;font-size:13px;cursor:pointer;transition:all .2s;
}
.quick-btn:active{background:#07c160;color:#fff}

/* ── 输入区 ── */
.input-area{display:flex;gap:8px;padding:10px 12px;background:#f7f7f7;border-top:1px solid #e0e0e0;flex-shrink:0}
.input-area input{flex:1;padding:10px 14px;border:1px solid #ddd;border-radius:8px;font-size:15px;outline:none;background:#fff}
.input-area input:focus{border-color:#07c160}
.send-btn{padding:0 18px;background:#07c160;color:#fff;border:none;border-radius:8px;font-size:15px;cursor:pointer;flex-shrink:0}
.send-btn:disabled{background:#a0d9a0;cursor:not-allowed}

/* ── 语音按钮 ── */
.voice-btn{
  width:40px;height:40px;border:1px solid #ddd;border-radius:50%;
  background:#fff;color:#666;font-size:18px;cursor:pointer;
  flex-shrink:0;display:flex;align-items:center;justify-content:center;
  transition:all .2s;
}
.voice-btn.recording{
  background:#ff4d4f;color:#fff;border-color:#ff4d4f;
  animation:pulse-ring 1.2s ease-in-out infinite;
}
@keyframes pulse-ring{
  0%{box-shadow:0 0 0 0 rgba(255,77,79,.5)}
  70%{box-shadow:0 0 0 10px rgba(255,77,79,0)}
  100%{box-shadow:0 0 0 0 rgba(255,77,79,0)}
}

/* ── 加载动画 ── */
.typing-dots{display:flex;gap:4px;padding:4px 0}
.typing-dots span{
  width:6px;height:6px;background:#999;border-radius:50%;
  animation:bounce .6s ease-in-out infinite;
}
.typing-dots span:nth-child(2){animation-delay:.15s}
.typing-dots span:nth-child(3){animation-delay:.3s}
@keyframes bounce{0%,80%,100%{transform:translateY(0)}40%{transform:translateY(-8px)}}

/* ══════ 简历预览弹窗 ══════ */
.modal-overlay{
  position:fixed;top:0;left:0;right:0;bottom:0;
  background:rgba(0,0,0,.6);z-index:100;
  display:none;align-items:flex-end;justify-content:center;
}
.modal-overlay.show{display:flex}
.modal{
  width:100%;max-width:500px;max-height:90vh;background:#fff;
  border-radius:16px 16px 0 0;display:flex;flex-direction:column;
  animation:slideUp .3s ease;
}
@keyframes slideUp{from{transform:translateY(100%)}to{transform:translateY(0)}}
.modal-header{display:flex;align-items:center;justify-content:space-between;padding:16px;border-bottom:1px solid #eee;flex-shrink:0}
.modal-header h3{font-size:16px;font-weight:600}
.modal-close{width:28px;height:28px;border:none;background:#f0f0f0;border-radius:50%;font-size:16px;cursor:pointer;color:#666;display:flex;align-items:center;justify-content:center}
.modal-body{flex:1;overflow-y:auto;-webkit-overflow-scrolling:touch}
.modal-footer{display:flex;gap:8px;padding:12px 16px;border-top:1px solid #eee;flex-shrink:0}
.modal-footer select{flex:1;padding:8px;border:1px solid #ddd;border-radius:8px;font-size:14px;background:#fff}
.modal-footer button{padding:8px 16px;border:none;border-radius:8px;font-size:14px;cursor:pointer}
.btn-download{background:#07c160;color:#fff}
</style>
</head>
<body>
<div class="app">
  <div class="header">
    <div class="header-left"><span class="header-title">📝 智聘Agent</span></div>
    <div class="header-right">
      <label class="tts-toggle" id="ttsToggle" onclick="toggleTTS()">
        <span>🔊 朗读</span><div class="tts-switch"></div>
      </label>
      <span id="completionText">0%</span>
      <div class="progress-bar"><div class="progress-fill" id="progressFill" style="width:0%"></div></div>
    </div>
  </div>

  <div class="toolbar">
    <div class="toolbar-btn" onclick="showPreview()">📄 预览简历</div>
    <div class="toolbar-btn" onclick="downloadResume()">⬇ 下载 PDF</div>
    <div class="toolbar-btn" onclick="resetChat()">🔄 重新开始</div>
  </div>

  <div class="chat-area" id="chatArea"></div>
  <div class="quick-replies" id="quickReplies"></div>

  <div class="input-area">
    <button class="voice-btn" id="voiceBtn" onclick="toggleVoice()" title="语音输入">🎤</button>
    <input type="text" id="userInput" placeholder="输入你的信息..." autocomplete="off">
    <button class="send-btn" id="sendBtn" onclick="sendMessage()">发送</button>
  </div>
</div>

<!-- 简历预览弹窗 -->
<div class="modal-overlay" id="previewModal">
  <div class="modal">
    <div class="modal-header">
      <h3>📄 简历预览</h3>
      <button class="modal-close" onclick="closePreview()">✕</button>
    </div>
    <div class="modal-body" id="previewBody"></div>
    <div class="modal-footer">
      <select id="templateSelect" onchange="refreshPreview()">
        <option value="classic_blue">经典蓝</option>
        <option value="business_dark">商务黑金</option>
        <option value="fresh_green">清新绿</option>
        <option value="minimal_gray">极简灰</option>
      </select>
      <button class="btn-download" onclick="downloadResume()">⬇ 下载 PDF</button>
    </div>
  </div>
</div>

<script>
// ══════ 初始化 ══════
const GREETING = `""" + GREETING.replace('`', '\\`').replace('\\n', '\n') + r"""`;

window.onload = function(){
  addMessage("assistant", GREETING);
  document.getElementById("userInput").addEventListener("keydown", function(e){
    if(e.key==="Enter"&&!e.shiftKey){e.preventDefault();sendMessage()}
  });
};

// ══════ 消息渲染 ══════
function escapeHtml(s){return s.replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;")}
function addMessage(role, content){
  const chatArea = document.getElementById("chatArea");
  const div = document.createElement("div");
  div.className = "msg " + role;
  const av = role==="assistant" ? "AI" : "我";
  div.innerHTML = `<div class="avatar">${av}</div><div class="bubble">${escapeHtml(content).replace(/\n/g,"<br>")}</div>`;
  chatArea.appendChild(div);
  setTimeout(()=>{chatArea.scrollTop=chatArea.scrollHeight},50);
}
function addTyping(){
  const chatArea = document.getElementById("chatArea");
  const div = document.createElement("div");
  div.className="msg assistant";div.id="typingMsg";
  div.innerHTML=`<div class="avatar">AI</div><div class="bubble"><div class="typing-dots"><span></span><span></span><span></span></div></div>`;
  chatArea.appendChild(div);
  setTimeout(()=>{chatArea.scrollTop=chatArea.scrollHeight},50);
}
function removeTyping(){const el=document.getElementById("typingMsg");if(el)el.remove()}

// ══════ 快捷回复 ══════
function renderQuickReplies(replies){
  const c=document.getElementById("quickReplies");c.innerHTML="";
  if(!replies||!replies.length)return;
  replies.forEach(text=>{
    const btn=document.createElement("div");btn.className="quick-btn";btn.textContent=text;
    btn.onclick=()=>{document.getElementById("userInput").value=text;sendMessage()};
    c.appendChild(btn);
  });
}

// ══════ 进度条 ══════
function updateProgress(val){
  document.getElementById("completionText").textContent=val+"%";
  document.getElementById("progressFill").style.width=val+"%";
}

// ══════ 发送消息（调用后端 /api/chat）══════
async function sendMessage(){
  const input=document.getElementById("userInput");
  const text=input.value.trim();
  if(!text)return;
  input.value="";
  document.getElementById("quickReplies").innerHTML="";
  addMessage("user",text);
  const sendBtn=document.getElementById("sendBtn");
  sendBtn.disabled=true;input.disabled=true;
  addTyping();
  try{
    const resp=await fetch("/api/chat",{
      method:"POST",headers:{"Content-Type":"application/json"},
      body:JSON.stringify({message:text})
    });
    const data=await resp.json();
    removeTyping();
    if(data.error){addMessage("assistant","抱歉，出了点问题："+data.error)}
    else{
      addMessage("assistant",data.reply);
      updateProgress(data.completion||0);
      renderQuickReplies(data.quick_replies||[]);
      speakText(data.reply);
    }
  }catch(err){removeTyping();addMessage("assistant","网络错误："+err.message)}
  sendBtn.disabled=false;input.disabled=false;input.focus();
}

// ══════ 预览简历（调用后端 /api/preview）══════
async function showPreview(){
  const tid=document.getElementById("templateSelect").value;
  const resp=await fetch("/api/preview",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({template:tid})});
  const data=await resp.json();
  document.getElementById("previewBody").innerHTML=
    `<iframe srcdoc='${data.html.replace(/'/g,"&#39;")}' style="width:100%;height:70vh;border:none"></iframe>`;
  document.getElementById("previewModal").classList.add("show");
}
async function refreshPreview(){await showPreview()}
function closePreview(){document.getElementById("previewModal").classList.remove("show")}

// ══════ 下载简历（调用后端 /api/download）══════
async function downloadResume(){
  const tid=document.getElementById("templateSelect").value;
  try{
    const resp=await fetch("/api/download",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({template:tid})});
    if(!resp.ok){const d=await resp.json();alert(d.error||"下载失败");return}
    const blob=await resp.blob();
    const a=document.createElement("a");
    a.href=URL.createObjectURL(blob);
    a.download=resp.headers.get("content-disposition")?.split("filename=")[1]||"简历.pdf";
    a.click();URL.revokeObjectURL(a.href);
  }catch(err){alert("下载失败："+err.message)}
}

// ══════ 重新开始 ══════
async function resetChat(){
  if(!confirm("确定要重新开始吗？"))return;
  await fetch("/api/reset",{method:"POST"});
  document.getElementById("chatArea").innerHTML="";
  document.getElementById("quickReplies").innerHTML="";
  updateProgress(0);
  addMessage("assistant",GREETING);
}

// ══════ 语音输入 ══════
let recognition=null,isRecording=false;
function initSpeech(){
  const SR=window.SpeechRecognition||window.webkitSpeechRecognition;
  if(!SR)return null;
  const r=new SR();r.lang="zh-CN";r.continuous=true;r.interimResults=true;
  let ft="",it="";
  r.onresult=function(e){ft="";it="";for(let i=0;i<e.results.length;i++){if(e.results[i].isFinal)ft+=e.results[i][0].transcript;else it+=e.results[i][0].transcript}document.getElementById("userInput").value=ft+it};
  r.onerror=function(e){if(e.error==="not-allowed")alert("请允许麦克风权限");stopRec()};
  r.onend=function(){if(isRecording)try{r.start()}catch(e){}};
  return r;
}
function toggleVoice(){if(isRecording){stopRec();const v=document.getElementById("userInput").value.trim();if(v)sendMessage()}else startRec()}
function startRec(){if(!recognition)recognition=initSpeech();if(!recognition){alert("浏览器不支持语音输入，建议用 Chrome");return}try{recognition.start();isRecording=true;document.getElementById("voiceBtn").classList.add("recording");document.getElementById("voiceBtn").textContent="⏹";document.getElementById("userInput").placeholder="正在听您说话..."}catch(e){alert("无法启动语音："+e.message)}}
function stopRec(){isRecording=false;if(recognition)try{recognition.stop()}catch(e){}document.getElementById("voiceBtn").classList.remove("recording");document.getElementById("voiceBtn").textContent="🎤";document.getElementById("userInput").placeholder="输入你的信息..."}

// ══════ TTS 自动朗读 ══════
let ttsEnabled=false;
function toggleTTS(){ttsEnabled=!ttsEnabled;const t=document.getElementById("ttsToggle");ttsEnabled?t.classList.add("active"):t.classList.remove("active");if(!ttsEnabled&&speechSynthesis.speaking)speechSynthesis.cancel()}
function getBestVoice(){const v=speechSynthesis.getVoices();const p=["Microsoft Xiaoxiao","Microsoft Yunxi","Tingting","Google","zh-CN"];for(const k of p){const f=v.find(x=>x.name.includes(k)||x.lang.includes(k));if(f)return f}return v.find(x=>x.lang.startsWith("zh"))||null}
function speakText(text){if(!ttsEnabled||!text)return;if(speechSynthesis.speaking)speechSynthesis.cancel();const c=text.replace(/[\u{1F300}-\u{1FAFF}]/gu,"").replace(/[「」【】▌①②③④⑤⑥⑦]/g,"").trim();if(!c)return;const u=new SpeechSynthesisUtterance(c);u.lang="zh-CN";u.rate=1;u.pitch=1;const v=getBestVoice();if(v)u.voice=v;speechSynthesis.speak(u)}
if(typeof speechSynthesis!=="undefined")speechSynthesis.onvoiceschanged=()=>speechSynthesis.getVoices();
</script>
</body>
</html>
"""

print(f"✅ 前端界面代码就绪（共 {len(FRONTEND_HTML)} 字符）")

In [ ]:
# ============================================================
# 注册根路由 — 返回前端页面
# 访问 http://localhost:5000 即可打开聊天界面
# ============================================================

@app.route("/")
def index():
    """返回前端聊天界面 HTML"""
    return Response(FRONTEND_HTML, mimetype="text/html")


print("✅ 根路由注册完成: GET /")
print("")
print("所有路由汇总：")
print("  GET  /             → 聊天界面")
print("  POST /api/chat     → 发送消息")
print("  POST /api/preview  → 预览简历")
print("  POST /api/download → 下载简历")
print("  POST /api/reset    → 重置对话")

---

## 第十一步：启动服务

运行下面的代码块即可启动 Flask 服务器。

启动后，在浏览器中打开 **http://localhost:5000** 即可使用智聘Agent。

> **注意事项：**
> - 在 Jupyter Notebook 中启动 Flask 后，当前 Notebook 的代码块将被阻塞（因为服务器在持续运行）
> - 如需停止服务，点击 Notebook 顶部的 **中断内核**（Interrupt Kernel）按钮
> - 使用 `use_reloader=False` 避免 Jupyter 环境下的重复启动问题
> - 使用 `threaded=True` 是为了让 Flask 在子线程中运行，这样在某些环境下 Notebook 仍可交互

In [ ]:
# ============================================================
# 启动 Flask 服务
# 运行后打开浏览器访问 http://localhost:5000
# 停止服务：点击 Jupyter 的「中断内核」按钮
# ============================================================

print("🚀 正在启动智聘Agent服务...")
print("")
print("═" * 50)
print("  智聘Agent - AI简历助手")
print("  访问地址: http://localhost:5000")
print("  停止服务: 点击 Jupyter 的「中断内核」按钮")
print("═" * 50)
print("")

# 启动 Flask（关闭自动重载，避免 Jupyter 冲突）
app.run(
    host="0.0.0.0",    # 允许局域网访问（手机同一WiFi下也能用）
    port=5000,          # 端口号
    debug=False,        # 关闭调试模式
    use_reloader=False  # 关闭自动重载（Jupyter 必须关闭）
)